In [2]:
%pip install great_expectations

  Using cached great_expectations-1.24.0-py3-none-any.whl.metadata (10 kB)
  Using cached marshmallow-4.3.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached tzlocal-5.4.4-py3-none-any.whl.metadata (7.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 34.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [great_expectations] 2/3 [great_expectations]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
debut = time.time()

import pandas as pd
import great_expectations as gx

erp = pd.read_excel('erp.xlsx')

# 1. Contexte éphémère : tout reste en mémoire, rien n'est écrit sur le disque
context = gx.get_context(mode="ephemeral")

# 2. Source de données, actif et lot
source = context.data_sources.add_pandas(name="bottleneck")
actif = source.add_dataframe_asset(name="erp")
definition = actif.add_batch_definition_whole_dataframe(name="erp_complet")
lot = definition.get_batch(batch_parameters={"dataframe": erp})

# 3. Suite d'attentes : R1, R2 et R4
suite = context.suites.add(gx.ExpectationSuite(name="regles_erp"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="price", min_value=0, strict_min=True))                      # R1
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="stock_quantity", min_value=0))                               # R2
suite.add_expectation(gx.expectations.ExpectColumnPairValuesAToBeGreaterThanB(
    column_A="price", column_B="purchase_price", or_equal=True))         # R4

# 4. Validation
resultat = lot.validate(suite)
print("Temps total :", round(time.time() - debut, 1), "s\n")

for r in resultat.results:
    print(r.expectation_config.type, "→ succès :", r.success,
          "| anomalies :", r.result.get("unexpected_count"))

Calculating Metrics:   0%|          | 0/23 [00:00<?, ?it/s]

Temps total : 2.0 s

expect_column_values_to_be_between → succès : False | anomalies : 3
expect_column_values_to_be_between → succès : False | anomalies : 2
expect_column_pair_values_a_to_be_greater_than_b → succès : False | anomalies : 7
